In [1]:
#! pip install transformers>=4.37.0 pillow optimum[onnxruntime]
from PIL import Image
from transformers import TrOCRProcessor
from optimum.onnxruntime import ORTModelForVision2Seq

processor = TrOCRProcessor.from_pretrained('breezedeus/pix2text-mfr')
model = ORTModelForVision2Seq.from_pretrained('breezedeus/pix2text-mfr', use_cache=False)




/home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Could not find any ONNX files with standard file name decoder_model_merged.onnx, files found: [PosixPath('decoder_model.onnx'), PosixPath('encoder_model.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


In [2]:
image_fps = [
    '../output/formulas/formula_page2_2.png',
    '../output/formulas/formula_page2_3.png',
    '../output/formulas/formula_page3_4.png'
]
images = [Image.open(fp).convert('RGB') for fp in image_fps]
pixel_values = processor(images=images, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
print(f'generated_ids: {generated_ids}, \ngenerated text: {generated_text}')

generated_ids: tensor([[  2,  95, 263, 346, 262, 331, 293, 293, 313, 284, 293, 281, 419, 284,
         261, 261, 267, 445, 271, 447, 271, 429, 266, 270, 262, 263, 346, 262,
         310, 419, 329, 293, 305, 300, 282, 261, 261, 263, 303,  12, 262, 263,
         277, 262, 445, 447, 265, 262, 359, 261, 261, 262, 263, 418, 262, 296,
         264, 262, 294, 261, 261, 261, 261, 263, 304,  13, 429, 271, 263, 549,
         740, 267, 269, 266,   2,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  2,  64, 422,  95, 412,  97, 262, 318, 261, 262, 262, 263, 346, 262,
         386, 328, 323, 379, 405, 313, 300, 296, 261, 267, 445, 271, 447, 271,
         429, 266, 270, 263, 346, 262, 398, 419, 284, 318, 300, 293, 261, 267,
         263, 346, 262, 379, 313, 300, 29

In [3]:
generated_text[0]


'{ \\mathrm { A t t e n t i o n } } ( Q, K, V ) = { \\mathrm { s o f t m a x } } \\left( { \\frac { Q K ^ { T } } { \\sqrt { d _ { k } } } } \\right) V, \\eqno ( 2 )'

In [4]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
llm= ChatGroq(model="openai/gpt-oss-120b")

In [7]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Variable(BaseModel):
    symbol: str = Field(..., description="The mathematical symbol used in the equation")
    name: str = Field(..., description="The name of the variable")
    details: str = Field(..., description="Detailed description of what the variable represents")

class FormulaOutput(BaseModel):
    description: str = Field(..., description="A simple explanation of the formula")
    variables: list[Variable] = Field(..., description="List of variables used in the equation")

In [8]:
# Create parser
parser = PydanticOutputParser(pydantic_object=FormulaOutput)

# Prompt
prompt_text = """
You are an expert mathematician who also knows latex code.
You are provided with following LaTeX code representing a mathematical equation.
{formula}

Your task is to:
1. Provide a simple explanation of the equation in layman's terms.
2. List and define all variables used in the equation with their symbol, name, and details.

{format_instructions}
"""
prompt = ChatPromptTemplate.from_template(prompt_text)

# Summary chain
model = ChatGroq(temperature=0.5, model="llama-3.1-8b-instant")
summarize_chain = prompt | model | parser

# Invoke the chain with the generated formula
result = summarize_chain.invoke({
    "formula": generated_text[0],
    "format_instructions": parser.get_format_instructions()
})
print(result)

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [ ]:
result.description

'The equation represents a formula for calculating attention based on three input values: Q, K, and V.'

In [ ]:
result.variables

[Variable(symbol='Q', name='Query', details='The query vector used in the attention calculation.'),
 Variable(symbol='K', name='Key', details='The key vector used in the attention calculation.'),
 Variable(symbol='V', name='Value', details='The value vector used in the attention calculation.'),
 Variable(symbol='d_k', name='Dimensionality of the Key', details='The dimensionality of the key vector K.')]